In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

# Đọc bộ dữ liệu 253.000 dòng của CDC
df = pd.read_csv('heart_disease_health_indicators_BRFSS2015.csv')

print("Kích thước dữ liệu:", df.shape)
# Kiểm tra xem có cột nào bị thiếu dữ liệu không (sẽ in ra 0 hết)
print("Số lượng dữ liệu thiếu:\n", df.isnull().sum().sum())
df.head()

In [ ]:
# Tách dữ liệu: y là đáp án (Cột Đột quỵ), X là các manh mối còn lại
X = df.drop('Stroke', axis=1)
y = df['Stroke']

# Vì 253.000 dòng là quá lớn, để AI học nhanh trên máy cá nhân, 
# ta sẽ cắt bớt lấy 100.000 dòng ngẫu nhiên. (Nếu máy bạn khỏe, có thể bỏ 3 dòng này)
df_sample = df.sample(n=100000, random_state=42)
X = df_sample.drop('Stroke', axis=1)
y = df_sample['Stroke']

print("Tỉ lệ Bệnh/Khỏe ban đầu:\n", y.value_counts())

In [ ]:
print("Đang nhân bản dữ liệu bệnh nhân bằng SMOTE. Vui lòng đợi khoảng 10-30 giây...")

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X, y)

print("✅ Đã cân bằng xong! Tỉ lệ mới:\n", y_balanced.value_counts())

In [ ]:
# Chia 80% để học, 20% để thi
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

print(f"Đang dạy AI trên {X_train.shape[0]} mẫu bệnh án...")

# Giả sử cột Rượu Bia nằm ở vị trí thứ 11 trong tập dữ liệu của bạn
rang_buoc = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

# Khởi tạo não bộ XGBoost (Cấu hình mạnh hơn một chút để xử lý dữ liệu lớn)
model = xgb.XGBClassifier(
    n_estimators=200,      # Đọc 200 cuốn sách quy luật
    max_depth=5,           # Mỗi cuốn sách phân tích sâu 5 tầng logic
    learning_rate=0.1,     # Tốc độ học chậm mà chắc
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
    monotone_constraints=rang_buoc # Tiêm kiến thức y khoa vào đây!
)

# Bắt đầu học!
model.fit(X_train, y_train)

print("✅ Quá trình huấn luyện hoàn tất!")

In [ ]:
# Cho AI làm bài thi cuối kỳ
y_pred = model.predict(X_test)

print("=== KẾT QUẢ ĐÁNH GIÁ AI TRÊN BỘ DỮ LIỆU CDC ===\n")
print(f"Độ chính xác tổng thể (Accuracy): {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Báo cáo chi tiết (Classification Report):")
print(classification_report(y_test, y_pred))

# Lưu não bộ thành file mới
joblib.dump(model, 'xgboost_cdc_stroke_model.pkl')
print("\nĐã lưu mô hình AI mới thành file 'xgboost_cdc_stroke_model.pkl'!")